In [ ]:
from pyspark.sql import SparkSession

In [ ]:
# Configuration de la session Spark
spark = SparkSession.builder \
    .appName("TP1-HelloDataLake") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio-datalake:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Session Spark initialisée avec succès !")

In [ ]:
# Lecture du fichier CSV
path_csv = "s3a://bronze/tpmilia/countries.csv"

In [ ]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv(path_csv)

In [ ]:
# Affichage des premières lignes
df.show()

Passer au format Parquet : Une fois votre fichier CSV lu depuis votre bucket (bronze pour l'exemple), essayez de l'écrire au format Parquet (plus performant) dans un autre bucket (silver ou gold par exemple)

In [ ]:
df.write.mode("overwrite").parquet("s3a://silver/tpmilia/countries/")

#### Vérifier le Spark UI : Allez jeter un œil sur http://127.0.0.1:4040 pendant que vous exécutez des calculs pour voir comment Spark planifie vos tâches.

En ingénierie des données (Data Engineering), on organise souvent le Data Lake en trois zones (Architecture en Médaillon) : Bronze (données brutes), Silver (données nettoyées et filtrées) et Gold (données agrégées pour les rapports).

## 1. Nettoyage de base (Standardisation)

Les fichiers CSV bruts contiennent souvent des valeurs manquantes (null), des espaces inutiles ou des colonnes mal nommées.Ajoutez cette cellule dans votre Notebook pour nettoyer vos données :

In [ ]:
from pyspark.sql.functions import col, trim, lower, when

In [ ]:
# 1. Nettoyer les espaces autour des noms de pays et continents
df_cleaned = df.withColumn("Country", trim(col("Country"))) \
               .withColumn("Continent", trim(col("Continent")))

In [ ]:
# 2. Remplacer les valeurs manquantes (Null) par 0 dans les colonnes numériques
df_cleaned = df_cleaned.fillna({"IMF_GDP": 0.0, "UN_GDP": 0.0, "Population": 0})

In [ ]:
# 3. Supprimer la ligne si le nom du pays est totalement absent
df_cleaned = df_cleaned.dropna(subset=["Country"])

In [ ]:
df_cleaned.show(5)

In [ ]:
df_cleaned.select("Country", "Continent", "Population", "IMF_GDP").show(5)

## 2. Transformation : Calculer l'écart entre le FMI (IMF) et l'ONU (UN)

Nous avons deux sources pour le PIB. Calculons l'écart en pourcentage entre l'estimation du FMI et celle de l'ONU pour voir quels pays affichent les plus grandes différences d'estimation.

In [ ]:
from pyspark.sql.functions import round, abs, when

In [ ]:
# Calcul de l'écart absolu et du pourcentage d'écart entre IMF_GDP et UN_GDP
df_transformed = df_cleaned.withColumn(
    "GDP_Difference", 
    round(abs(col("IMF_GDP") - col("UN_GDP")), 2)
).withColumn(
    "GDP_Diff_Percent",
    # Évite la division par zéro si UN_GDP est nul
    when(col("UN_GDP") > 0, round((col("GDP_Difference") / col("UN_GDP")) * 100, 2)).otherwise(0.0)
)

In [ ]:
# Filtrer pour voir uniquement les pays d'Afrique (au vu de notre localisation)
df_africa = df_transformed.filter(col("Continent") == "Africa") \
                          .orderBy(col("GDP_per_capita").desc())

In [ ]:
df_africa.select("Country", "Population", "GDP_per_capita", "GDP_Diff_Percent").show(10)

## 3. Partitionnement et stockage dans le Data Lake (Zone Silver)

Nous allons maintenant enregistrer ce résultat nettoyé dans MinIO au format Parquet. Nous allons créer des sous-dossiers automatiques pour chaque continent grâce à partitionBy.

In [ ]:
# Créez au préalable le bucket "silver" dans l'interface MinIO (http://localhost:9001)
target_path = "s3a://silver/tpmilia/countries_analyzed"

In [ ]:
print("Enregistrement des données partitionnées par Continent dans MinIO...")

df_transformed.write \
    .mode("overwrite") \
    .partitionBy("Continent") \
    .parquet(target_path)

print("Données sauvegardées avec succès au format Parquet !")

## 4. Agrégation : Statistiques globales par Continent (Zone Gold)

Créons une vue synthétique pour votre rapport : la population totale, le PIB cumulé (FMI) et le PIB par habitant moyen pour chaque continent.

In [ ]:
from pyspark.sql.functions import sum, avg, count

In [ ]:
df_gold_continents = df_transformed.groupBy("Continent").agg(
    count("Country").alias("Number_of_Countries"),
    sum("Population").alias("Total_Population"),
    round(sum("IMF_GDP"), 2).alias("Total_IMF_GDP"),
    round(avg("GDP_per_capita"), 2).alias("Average_GDP_per_capita")
).orderBy(col("Total_IMF_GDP").desc())

In [ ]:
df_gold_continents.show()

## 5. Exécuter du SQL pur dans Spark

Pour faire du SQL dans Spark, il faut d'abord enregistrer le DataFrame comme une vue temporaire. Cela permet à Spark de comprendre le nom de la table dans la requête SELECT.

Voici comment l'appliquer sur votre dataset de pays (df_transformed) 

In [ ]:
# 1. Enregistrer le DataFrame comme une table SQL nommée "countries_table"
df_transformed.createOrReplaceTempView("countries_table")

In [ ]:
# Exemple 1 : Requête simple avec filtre et tri
query_sql_1 = """
    SELECT Country, Continent, Population, GDP_per_capita 
    FROM countries_table 
    WHERE Continent = 'Africa' AND GDP_per_capita > 2000
    ORDER BY GDP_per_capita DESC
"""

In [ ]:
# Exécution de la requête
df_result_1 = spark.sql(query_sql_1)

In [ ]:
df_result_1.show(5)

In [ ]:
# Exemple 2 : Agrégation complexe (Top 3 des continents par PIB total)
query_sql_2 = """
    SELECT Continent, COUNT(*) as Total_Countries, SUM(IMF_GDP) as Total_GDP
    FROM countries_table
    GROUP BY Continent
    ORDER BY Total_GDP DESC
    LIMIT 3
"""

In [ ]:
spark.sql(query_sql_2).show()